# Setting Up

In [ ]:
import torch
import numpy as np
import pandas as pd
import random
from torch.utils.data import DataLoader, Dataset, DistributedSampler, IterableDataset
import os

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [97]:
!nvidia-smi

Sat Jul 25 22:19:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.74                 KMD Version: 610.74        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   47C    P8              3W /  160W |       0MiB /  12282MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Dataset

## Simple Approach

#### Most of the time in research we will have to create a custom Dataset class. The necessary functions and attributes are:
- __init__(self, ...): The Setup. This is where you pass in your raw data (or file paths). You store them as class attributes here.

- __len__(self): The Size. This function must return a single integer representing the total number of samples in your dataset. The DataLoader uses this to know when an epoch is over.

- __getitem__(self, idx): The Fetcher. This is the heart of the dataset. It takes an integer idx (index) and must return exactly one preprocessed sample (features and labels) converted to PyTorch Tensors.

In [98]:
dummy_data = {
    "sensor_1": np.random.rand(1000),
    "sensor_2": np.random.rand(1000),
    "sensor_3": np.random.rand(1000),
    "action_label": np.random.randint(0, 4, 1000) # 4 discrete RL actions
}
raw_df = pd.DataFrame(dummy_data)

In [99]:
class InMemoryDataset(Dataset):
    def __init__(self, data: pd.DataFrame, label_col: str):
        raw_features = data.drop(columns=[label_col]).to_numpy()
        raw_labels = data[label_col].to_numpy()

        self.features = torch.from_numpy(raw_features).float() # from_numpy converts numpy arrays to torch tensors, saves memory
        self.labels = torch.from_numpy(raw_labels).long()

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return {'features': self.features[idx], 'label': self.labels[idx]}
        

In [100]:
baseline_dataset = InMemoryDataset(raw_df, label_col="action_label")
sample = baseline_dataset[0]

print(f"\nSample keys: {sample.keys()}")
print(f"Features shape: {sample['features'].shape}, dtype: {sample['features'].dtype}")
print(f"Label value: {sample['label']}, dtype: {sample['label'].dtype}")


Sample keys: dict_keys(['features', 'label'])
Features shape: torch.Size([3]), dtype: torch.float32
Label value: 3, dtype: torch.int64


## The Scaling Bottleneck

### Proving the RAM Crash
Our Pandas-based `RealisticInMemoryDataset` works perfectly for a CSV with 10,000 rows. But what happens if we transition to Reinforcement Learning or Computer Vision? 

At a frontier lab, you might be training an agent on **1 Million Atari frames** or high-resolution MuJoCo camera feeds. Let's write a quick script to calculate exactly what happens to our system RAM if we try to load that dataset using our baseline approach.

In [101]:
def calculate_ram_usage(num_samples: int, shape: tuple, dtype_bytes: int = 4):
    """Calculates the theoretical RAM required to hold a dataset in memory."""
    elements_per_sample = 1
    for dim in shape:
        elements_per_sample *= dim
        
    total_bytes = num_samples * elements_per_sample * dtype_bytes
    return total_bytes / (1024 ** 3) # Convert to Gigabytes

# 1. Our previous CSV dataset (1,000 rows of 3 sensor features)
csv_gb = calculate_ram_usage(num_samples=1_000, shape=(3,))
print(f"RAM for 1k CSV rows: {csv_gb:.6f} GB (Safe!)")

# 2. A standard RL Vision Dataset (1 Million RGB Frames at 256x256)
# Shape: [Channels(3), Height(256), Width(256)]
vision_gb = calculate_ram_usage(num_samples=1_000_000, shape=(3, 256, 256))
print(f"RAM for 1M Atari frames: {vision_gb:.2f} GB (Unsafe!)")

RAM for 1k CSV rows: 0.000011 GB (Safe!)
RAM for 1M Atari frames: 732.42 GB (Unsafe!)


## The Scaling Fix (Lazy Loading)

**The Evidence:** Trying to load 1 million images into `__init__` requires **over 732 GB of RAM**. The Python process will instantly crash with an `Out Of Memory (OOM)` error.

The solution is Lazy Loading: 
1. **The Setup (`__init__`)**: We stop loading the actual data (payloads). Instead, we only load a list of string paths or memory indices (pointers).
2. **The Fetch (`__getitem__`)**: The heavy lifting is deferred here. We open the file from disk, convert it to a tensor, and return it *only when the DataLoader specifically asks for it*.

In [102]:
class LazyLoadDataset(Dataset):
    def __init__(self, paths:list[str], labels:list[int]):
        self.paths = paths
        self.labels = labels

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        # if each file is a single sample, this will work. If each file contains multiple samples, you will need to modify this logic accordingly.
        sample_path = self.paths[idx]
        sample_data = self.simulate_file_read(sample_path)  # Simulate reading the file
        return {'features': torch.tensor(sample_data, dtype=torch.float32),
                 'label': torch.tensor(self.labels[idx], dtype=torch.long)}
    def simulate_file_read(self, path):
        # Simulate reading a file and returning its content as a numpy array
        # In practice, you would read the actual file here.
        return np.random.rand(3, 256, 256)  # Simulating an RGB image of shape (3, 256, 256)
    

In [103]:
dummy_paths = [f"data/frame_{i}.png" for i in range(1_000_000)]
dummy_labels = [np.random.randint(0, 4) for _ in range(1_000_000)]


In [104]:
lazy_dataset = LazyLoadDataset(dummy_paths, dummy_labels)
sample = lazy_dataset[0]
sample['features'].shape, sample['label']

(torch.Size([3, 256, 256]), tensor(0))

### The Multiprocessing Trap (Safe HDF5 Pointers)

In RL and large-scale Vision, datasets are often packed into single massive files (like HDF5 or LMDB) for fast sequential reading. 

If you open the `h5py.File` inside the `__init__` method, that single OS file pointer is copied to all background CPU workers when the DataLoader forks the process. Multiple CPU cores trying to read from the exact same file pointer simultaneously will cause corrupted reads or silent deadlocks.

**The Production Fix (Lazy Init):**
We must wait to open the file until the background worker actually executes `__getitem__` for the first time. This ensures every worker process gets its own isolated, thread-safe connection to the database.


In [105]:
class SafeDatabaseDataset(Dataset):
    def __init__(self, db_path: str, num_samples: int):
        self.db_path = db_path
        self.num_samples = num_samples

        # AVOID THIS HERE
        # self.db = h5py.File(self.database_path, 'r')

        self.db=None

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):

        if self.db is None:
            # This is important because this way we don't need to open a connection everytime we read a sample. Instead, we open it once per worker process.
            self.db = self.open_database()

            worker_id = os.getpid()
            print(f"[Worker {worker_id}] established an isolated database connection.")

        # Simulate reading a row from the database
        raw_state = [0.1, 0.2, 0.3] # Imagine: self.db['states'][idx]

        return {
            "state": raw_state,
            "worker_pid": os.getpid() # For debugging
        }
        

## On The Fly Transformations (Augmentation)

### The Hard Drive Explosion

To prevent neural networks from memorizing training data, researchers use augmentation (e.g., adding Gaussian noise to RL state vectors, or randomly cropping images). 

**The Beginner Mistake (Static Augmentation):** A beginner might write a script that generates 5 augmented variations of every file and saves them all to the hard drive. 
* **The Math:** If your base dataset is 100 GB, it just became 600 GB. 
* **The Reality:** The neural network will eventually memorize those exact 5 static variations anyway.

**The Production Fix (On-The-Fly Transformations):**
Instead of saving variations to disk, we pass a callable function (a `transform`) into the Dataset's `__init__`. Inside `__getitem__`, *after* loading the raw data but *before* returning it to the DataLoader, we apply the function. This provides **infinite, non-repeating variations** while using **zero extra hard drive space**.

transform function can be used to transform data for any use. You can for instance use it for offline preprocessing (cleaning, feature engineering, etc.) or in our case right now we will be using it for online transformations (augmentation).

If your online transforms ever get too mathematically heavy and start slowing down the GPU, the advanced PyTorch move is to push the transforms out of the CPU Dataset entirely and execute them directly on the GPU batch using libraries like Kornia or torchvision.transforms.v2.

### Custom Transform

In [106]:
class AddGaussianNoise:
    def __init__(self, mean: float = 0.0, std: float = 0.1):
        self.mean = mean
        self.std = std

    def __call__(self, tensor: torch.Tensor) -> torch.Tensor:
        # torch.randn_like creates noise with the exact same shape and device as the input
        noise = torch.randn_like(tensor) * self.std + self.mean
        return tensor + noise

In [107]:
class TransformedDataset(Dataset):
    def __init__(self,  paths:list[str], labels:list[int], transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform  # Store the transform function

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        sample_path = self.paths[idx]

        sample_data = torch.tensor(self.simulate_file_read(sample_path), dtype=torch.float32)  # Simulate reading the file
        if self.transform:
            sample_data = self.transform(sample_data)

        return {'features': sample_data,
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}

    def simulate_file_read(self, path):
        # Simulate reading a file and returning its content as a numpy array
        # In practice, you would read the actual file here.
        return np.ones((3, 256, 256))  # Simulating an RGB image of shape (3, 256, 256)
            
        

In [108]:
noise_transform = AddGaussianNoise(std=0.15)
dataset = TransformedDataset(paths=dummy_paths, labels=dummy_labels, transform=noise_transform)
dataset[0]['features'][0]==dataset[0]['features'][0] # will return all False because we havent set the random seed

tensor([[False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        ...,
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False]])

When using random augmentations with multiple workers (num_workers > 0), be careful with random seeds. It’s best to use PyTorch’s random functions (torch.rand, etc.) or properly seed each worker in worker_init_fn to ensure reproducibility or the desired randomness.

Some argue some argue that moving augmentations after the DataLoader (e.g., using Kornia on the batch in the training loop or using torch.nn.Sequential) can simplify the Dataset code and might be more stable/performant in multiprocessing setups.


### Metadata Injection for Traceability (Debugging)

You are training a complex model or Reinforcement Learning agent for three days straight. Suddenly, on epoch 450, your neural network loss spikes to `NaN` (Not a Number) or crashes with an unhandled shape mismatch. 

If your `__getitem__` only returns standard features and labels (`{"state": tensor, "action": tensor}`), your training loop is completely blind. You have no idea *which* file, *which* episode index, or *which* timestep caused the corruption that blew up your model.

**The Production Fix:**
A research-grade dataset should always inject a `metadata` dictionary into its return payload. This acts as a transparent tracking log. If a batch causes an exception, you can instantly catch it and print the exact file path or index that triggered the bug.

In [109]:
class TraceableDataset(Dataset):
    def __init__(self, file_paths: list[str]):
        self.file_paths = file_paths

    def __len__(self) -> int:
        return len(self.file_paths)

    def __getitem__(self, idx: int):
        target_file = self.file_paths[idx]
        
        # Simulate loading raw array (e.g., state vector or observation)
        raw_data = np.array([0.5, -1.2, 3.4])
        
        feature_tensor = torch.tensor(raw_data, dtype=torch.float32)
        label_tensor = torch.tensor(1, dtype=torch.int64) # Dummy label/action
        
        # Pack the actual model tensors alongside lightweight debugging context
        return {
            "features": feature_tensor,
            "label": label_tensor,
            "metadata": {
                "sample_index": idx,
                "file_source": target_file
            }
        }

In [110]:
# --- Testing the Traceability ---
dummy_paths = [f"data/episodes/run_alpha_step_{i}.h5" for i in range(50)]
dataset = TraceableDataset(dummy_paths)

# Fetch a single sample
sample = dataset[42]

print(f"Features: {sample['features']} (dtype: {sample['features'].dtype})")
print(f"Label:    {sample['label']} (dtype: {sample['label'].dtype})")

print(f"Sample Index: {sample['metadata']['sample_index']}")
print(f"Source File:  {sample['metadata']['file_source']}")

Features: tensor([ 0.5000, -1.2000,  3.4000]) (dtype: torch.float32)
Label:    1 (dtype: torch.int64)
Sample Index: 42
Source File:  data/episodes/run_alpha_step_42.h5


## When to use IterableDataset (Streaming)

Everything built above is a Map-Style Dataset (`__getitem__`), which requires knowing the total dataset length. But what if the data is infinite, or streaming over a network from AWS S3? Randomly accessing indices over a network connection is far too slow.

For distributed Reinforcement Learning (where agents stream live transitions over a network to a central learner) or massive cloud datasets, researchers switch to **`IterableDataset`**. 
Instead of asking for a specific index, the DataLoader simply asks for the `next()` item in a continuous stream.

In [111]:

class StreamingDataset(IterableDataset):
    def __init__(self, stream_url: str):
        self.stream_url = stream_url
        
    def _live_environment_stream(self):
        """Simulates an infinite stream of data from a network."""
        step = 0
        while True:
            yield {
                "state": torch.rand(3, dtype=torch.float32),
                "timestep": step
            }
            step += 1
            if step > 3: break # Just for this test to not run forever

    def __iter__(self):
        # Instead of __getitem__, we return an iterator that yields data sequentially
        return self._live_environment_stream()

stream_dataset = StreamingDataset("wss://lab-server/stream")

stream_iterator = iter(stream_dataset)

print("--- Streaming Live Data ---")
print(f"Fetch 1: {next(stream_iterator)}")
print(f"Fetch 2: {next(stream_iterator)}")

--- Streaming Live Data ---
Fetch 1: {'state': tensor([0.9290, 0.5637, 0.7897]), 'timestep': 0}
Fetch 2: {'state': tensor([0.4210, 0.8788, 0.6428]), 'timestep': 1}


# Data Loader

### Our Best Practice Dataset

In [112]:
class ObservationDataset(Dataset):
    def __init__(self, file_paths: list, transform=None):
        super().__init__()
        self.file_paths = file_paths
        self.transform = transform
        self.db = None

    def __len__(self) -> int:
        return len(self.file_paths)

    def __getitem__(self, idx: int) -> dict:
        # Lazy initialization for safe CPU worker connections
        if self.db is None: 
            self.db = "Database_Connection_Active" 

        target_file = self.file_paths[idx]
        
        # Simulate reading an RL image state [Channels, Height, Width] from disk
        raw_state = np.random.randn(3, 84, 84) 
        
        # Strict Type Enforcement
        state_tensor = torch.tensor(raw_state, dtype=torch.float32)
        label_tensor = torch.tensor(1, dtype=torch.int64) 
        
        # On-the-fly Transform
        if self.transform:
            state_tensor = self.transform(state_tensor)
            
        # Dictionary Return & Metadata Injection
        return {
            "state": state_tensor,
            "action": label_tensor,
            "metadata": {"original_index": idx, "worker_pid": os.getpid()}
        }

## Simple DataLoader

In [113]:
dummy_paths = [f"data/rollout_{i}.h5" for i in range(1000)]

# Initialize our dataset
dataset = ObservationDataset(dummy_paths)

In [114]:
# Initialize the Simple DataLoader
naive_dataloader = DataLoader(
    dataset=dataset, 
    batch_size=32,   # Group 32 samples together
    shuffle=True     # Randomize the order every epoch
)

In [115]:
first_batch = next(iter(naive_dataloader))
first_batch
print(f"Batched State Shape:  {first_batch['state'].shape}  <- new '32' at dimension 0!")
print(f"Batched Action Shape: {first_batch['action'].shape}")
print(f"Data Types: State is {first_batch['state'].dtype}, Action is {first_batch['action'].dtype}")

Batched State Shape:  torch.Size([32, 3, 84, 84])  <- new '32' at dimension 0!
Batched Action Shape: torch.Size([32])
Data Types: State is torch.float32, Action is torch.int64


Notice that our `ObservationDataset` only returned a state of shape `[3, 84, 84]`. The `DataLoader` automatically stacked 32 of them together to create a shape of `[32, 3, 84, 84]`. 

However, if you check the profiler, this batch was extremely slow to build. Because we didn't specify `num_workers`, the main Python process had to execute `__getitem__` 32 separate times sequentially before it could finally hand the `[32, 3, 84, 84]` tensor to the GPU.

## CPU Bottleneck (Multiprocessing)

In our baseline `DataLoader`, the main Python process was forced to sequentially fetch 32 individual images from the hard drive before it could pass the batch to the GPU. This starves the GPU, leaving it sitting idle at 0% utilization (the "Sawtooth" bottleneck).

Tto fix it we have to do multiple things:
1. **`num_workers>0`**: PyTorch will use Python's multiprocessing to fork n independent background CPU processes. These workers will constantly fetch and batch the *next* iterations of data while the GPU is busy doing math on the *current* iteration.
2. **`persistent_workers=True`**: Normally, PyTorch destroys all background workers at the end of every epoch and respawns them for the next one. This can cause a massive 30-second delay between epochs. Setting this to `True` keeps the OS processes alive, ensuring Epoch 2 starts in exactly 0.01 seconds.
3. **`drop_last=True`**: If your dataset has 100 items and your batch size is 32, your final batch will only have 4 items. This sudden shape change often crashes Batch Normalization layers. `drop_last` tells the DataLoader to safely discard that incomplete final batch.

In [ ]:
# We MUST wrap the execution in this block so the spawned workers don't recursively run it
# Note This will give an error if you run it in a Jupyter Notebook, but it will work in a standard Python script because of Windows.
if __name__ == '__main__':

    optimized_dataloader = DataLoader(
        dataset=dataset,             
        batch_size=32,
        shuffle=True,
        num_workers=4,
        persistent_workers=True,
        drop_last=True               
    )

    # Fetch a batch to force the workers to spawn and read
    optimized_batch = next(iter(optimized_dataloader))
    print(f"Workers alive: {optimized_dataloader.num_workers}")
    print(f"State Shape: {optimized_batch['state'].shape}")

#### How to choose num of workers

Like many things in ML, it is by experimentation, but the rule of thumb is:
- Optimal Workers = min(4 * GPUs, CPU_Cores)

CPU is the limit to avoid context switching, and 4 is because empirically, across standard vision and RL tasks, it takes the combined effort of about 3 to 4 CPU cores running at maximum speed to prep data fast enough to satisfy 1 modern GPU.

But this isnt a strict law, you may need for heavy augmentatinon 8 workers per GPU but for another problem with light data you may need 1 or 2

## Utilizing Full Hardware Capabilities

We fixed the CPU bottleneck using multiprocessing, but we still have two hardware chokepoints:
1. **The PCIe Bus (Motherboard Transfer):** When we call `batch.to('cuda')`, the data moves from system RAM to GPU VRAM. Standard RAM is "pageable" (the OS can move it around), meaning the transfer is slow. 
   * *Fix:* We set `pin_memory=True`. This locks the data in a specific RAM sector, allowing the GPU to use DMA (Direct Memory Access) to suck the data over the PCIe bus incredibly fast.
2. **I/O Latency Spikes:** Sometimes a hard drive takes an extra millisecond to read a file. 
   * *Fix:* We use `prefetch_factor=2`. This forces our background workers to always keep a buffer of `(num_workers * prefetch_factor)` batches pre-loaded in RAM, completely hiding disk latency.

In [117]:
if __name__ == '__main__':    
    ultimate_dataloader = DataLoader(
        dataset=dataset,             
        batch_size=32,
        shuffle=True,
        num_workers=4,               
        persistent_workers=True,  
        drop_last=True,
        
        # THE HARDWARE OPTIMIZATIONS 
        # 1. Lock memory for fast PCIe transfer to GPU
        pin_memory=True, 
        
        # 2. Instruct workers to queue up 2 batches EACH (8 batches total in RAM)
        # Note: prefetch_factor can ONLY be used if num_workers > 0
        prefetch_factor=4 
    )

#### What are the trade-offs?

We discussed num_workers with context switching.

Pinning Memory is good most of the times, unless the data is large and you starve your operating system from memory

this dictates how many batches each worker holds in RAM. If a batch is 2GB, 4 workers with a prefetch of 5 means 40GB of RAM is held hostage. The worst part? A prefetch of 5 doesn't make the GPU run any faster than a prefetch of 2; it just wastes memory holding food the GPU isn't ready to eat yet. Just leave it at the PyTorch default unless the harddrive is way too slow.

With Persistent Workers, If you have a tiny "memory leak" in your __getitem__ function (e.g., appending to a global list without clearing it), normally it gets wiped out when the epoch ends and the workers are destroyed. With persistent_workers=True, that memory leak is kept alive permanently. Your RAM usage will slowly creep up over 5 hours until the script crashes. Normally keep it on but keep your Dataset Code clean

## Batch Processing

When dealing with variable-length sequences (text sentences or RL episodes), we cannot use PyTorch's default stacking logic. We must write a custom `collate_fn`,a function that tells the DataLoader exactly how to stitch a list of individual samples together.

Our `collate_fn` will do two things:
1. **Dynamic Padding:** Find the longest episode in the *current batch* and pad all shorter episodes with zeros until they match that max length.
2. **Attention Masking:** Generate a binary tensor (1s and 0s) of the exact same shape. This mask tells the neural network (like an LSTM or Transformer) to completely ignore the fake padded zeros when calculating the loss.

In [118]:
from torch.nn.utils.rnn import pad_sequence
class RLDynamicBatchCollator:
    def __init__(self, pad_value=0.0):
        self.pad_value = pad_value

    def __call__(self, batch: list) -> dict:
        """
        This function receives a list of dictionaries (one for each sample).
        Example: [{"state": tensor(14, 4), "length": 14}, {"state": tensor(22, 4), "length": 22}]
        """
        # Extract all state tensors and lengths from the list of dictionaries
        states = [item["state"] for item in batch]
        lengths = [item["length"] for item in batch]
        
        # 1. PAD THE SEQUENCES
        # pad_sequence automatically finds the longest tensor in this specific list
        # batch_first=True ensures the output is [Batch, Time, Features] instead of [Time, Batch, ...]
        padded_states = pad_sequence(states, batch_first=True, padding_value=self.pad_value)
        
        # 2. GENERATE THE MASK
        batch_size = len(states)
        max_len = padded_states.size(1) # The length of the longest episode
        
        # Create a boolean matrix where True = Real Data, False = Padded Zero
        # (This uses broadcasting to compare all indices against the real lengths)
        #### arange create a tensor list 0 to max_len-1, then we basically duplicate that list using expand(shape)
        #### so we have for each sample in the batch a list of 0 to max_len-1, then we compare that list with the actual length of each sample
        #### we use unsqueeze to make it broadcastable, basically each length is a list of its own, so we can compare it with the list of 0 to max_len-1
        #### less than the length means that we are in the real data, greater than or equal to the length means we are in the padded data
        mask = torch.arange(max_len).expand(batch_size, max_len) < torch.tensor(lengths).unsqueeze(1)

        
        return {
            "state": padded_states,
            "mask": mask,
            "original_lengths": torch.tensor(lengths)
        }

class VariableLengthRLDataset(Dataset):
    def __init__(self, num_episodes=100):
        self.num_episodes = num_episodes

    def __len__(self):
        return self.num_episodes

    def __getitem__(self, idx):
        # Simulate an RL episode of RANDOM length between 10 and 50 steps
        episode_length = torch.randint(low=10, high=50, size=(1,)).item()
        
        # State shape: [TimeSteps, Sensor_Features]
        states = torch.randn(episode_length, 4) 
        
        return {
            "state": states,
            "length": episode_length # Track the true length for debugging
        }

In [119]:
rl_dataset = VariableLengthRLDataset()
rl_dataloader = DataLoader(rl_dataset, batch_size=4, shuffle=True)

rl_dataloader = DataLoader(
    rl_dataset, 
    batch_size=4, 
    shuffle=True,
    collate_fn=RLDynamicBatchCollator(pad_value=0.0) 
)
batch = next(iter(rl_dataloader))

print(f"Original Lengths in this batch: {batch['original_lengths'].tolist()}")
print(f"Padded State Shape: {batch['state'].shape}  <-- Notice the time dimension perfectly matches the max length!")
print(f"Padding Mask Shape: {batch['mask'].shape}")

# Look at the mask for the shortest episode to prove it worked
shortest_idx = torch.argmin(batch['original_lengths']).item()
print(f"\nMask for the shortest episode (Length {batch['original_lengths'][shortest_idx]}):")
print(batch['mask'][shortest_idx])

Original Lengths in this batch: [24, 19, 31, 22]
Padded State Shape: torch.Size([4, 31, 4])  <-- Notice the time dimension perfectly matches the max length!
Padding Mask Shape: torch.Size([4, 31])

Mask for the shortest episode (Length 19):
tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True, False,
        False, False, False, False, False, False, False, False, False, False,
        False])


## Reproducability

When PyTorch forks background CPU workers (especially on Linux), it copies the exact state of the main process. This means **every single worker gets the exact same Random Number Generator (RNG) seed.** If you use `numpy.random` or Python's built-in `random` module to apply noise or explore environments in your `__getitem__`, all 4 workers will generate the *exact same random numbers*. Your data won't actually be random!
* The Fix: We must pass a `worker_init_fn` to the DataLoader to manually re-seed Numpy for each specific worker ID.

In [120]:
def seed_worker(worker_id):
    """
    This function is called once per worker when it spawns.
    It mathematically guarantees that Worker 1 and Worker 2 get completely 
    different random seeds, preventing identical augmentations.
    """
    # Create a unique seed based on the base seed + the worker's unique ID
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [121]:
DataLoader(
    rl_dataset, 
    batch_size=32, 
    num_workers=4,
    pin_memory=True,
    collate_fn=RLDynamicBatchCollator(pad_value=0.0),
    
    worker_init_fn=seed_worker,  
)

## Multiple GPUs

In a research lab, you don't train on 1 GPU; you train on an 8-GPU node. If you just pass a standard `DataLoader` to all 8 GPUs, they will all read the exact same data and do redundant math.
* The Fix: We use a `DistributedSampler`. It mathematically chunks the dataset so that GPU 0 only sees indices 0-100, GPU 1 sees 101-200, etc., ensuring no data overlap.

When moving to a Multi-GPU cluster, we inject a `DistributedSampler` into our DataLoader.
1. You MUST set `shuffle=False` in the `DataLoader`. The Sampler takes over shuffling duties.
2. You MUST call `sampler.set_epoch(epoch)` inside your training loop, otherwise your GPUs will process the exact same sequence of files every single epoch.

In [ ]:
# Assume this is running on a massive lab server with 4 GPUs
WORLD_SIZE = 4 
CURRENT_GPU_ID = 0 # (Rank 0) # you will need to set this dynamically for each GPU process in a real DDP setup for example: CURRENT_GPU_ID = int(os.environ["LOCAL_RANK"])

# Initialize the Sampler
multi_gpu_sampler = DistributedSampler(
    dataset=rl_dataset,
    num_replicas=WORLD_SIZE,  
    rank=CURRENT_GPU_ID,      
    shuffle=True,             
    drop_last=False # Padding is active
)

# Initialize the DataLoader (Notice shuffle=False)
ddp_dataloader = DataLoader(
    dataset=rl_dataset, 
    batch_size=32, 
    sampler=multi_gpu_sampler, # <- Sampler takes control
    shuffle=False,             # <- MUST BE FALSE
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

num_epochs = 5

for epoch in range(num_epochs):
    # This guarantees the mathematical shuffle changes every epoch across all GPUs
    multi_gpu_sampler.set_epoch(epoch)
    
    # Now you can safely iterate over your dataloader
    # for batch in ddp_dataloader:
    #     train_step(batch)

# Identifying Bottlenecks with Pytorch Profiler

The golden rule of deep learning systems is: Never let the GPU wait.

If our GPU utilization is low, we need to know if the CPU is struggling to load data from disk, or if the CPU-to-GPU memory transfer (PCIe bus) is the bottleneck.

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity
import torch.nn as nn

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = nn.Linear(1000, 10).to(device)
dummy_dataloader = [(torch.randn(64, 1000), torch.empty(64, dtype=torch.int64).random_(10)) for _ in range(5)]

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    
    for inputs, labels in dummy_dataloader:
        # Step A: Data Transfer (Are we blocked by the PCIe bus?)
        with record_function("1_data_transfer"):
            inputs, labels = inputs.to(device), labels.to(device)
            
        # Step B: Forward Pass (Are we compute bound?)
        with record_function("2_forward_pass"):
            outputs = model(inputs)
            
print("\n--- Profiler Results ---")
# Sort by CUDA time to see what is dominating the GPU
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

Warming up...

--- Profiler Results ---
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                         2_forward_pass         0.00%       0.000us         0.00%       0.000us       0.000us       4.971ms      3278.05%       4.971ms     994.166us           0 B 